In [197]:
import pandas as pd
import numpy as np
import re

In [198]:
# tabela = pd.read_excel()

In [199]:
import re

# Padrões com \d+ para representar qualquer número
#Padronizar os dias
def verificar_padrao(entry):
    text = str(entry)
    padroes = [
    r"^\d+\s*a\s*\d+$",
    r"^\d+\s*-\s*\d+$",
    r"^\d+\s*-\s*\d+\s*dias$",
    r"^de\s+\d+\s*a\s+\d+\s*dias$",
    r"^\d+\s*e\s*\d+\s*dias$",
    r"^acima\s+de\s+\d+\s*dias$",
    r"^superior\s+a\s+\d+$",
    r"^superior\s+\d+$",
    r"^>\s*\d+$",
    r"^até\s+\d+$",
    r"^até\s+\d+\s*dias$"
    ]
    text = text.lower().strip()
    return any(re.search(padrao, text) for padrao in padroes)

def verificar_padrao2(entry):
    text = str(entry).lower().strip()
    padroes = [
        r'(\d+)\s*-\s*(\d+)\s*dias?',                   # intervalo: 10 - 20 dias
        r'>\s*(\d+)\s*dias?',                           # > 120 dias
        r'<=\s*(\d+)\s*dias?'                           # <= 10 dias
    ]
    return any(re.search(p, text) for p in padroes)

def padronizar_string(texto):
    if not texto or not isinstance(texto, str):
        return None

    texto = texto.lower().strip()
    subtrair_se_comecar_em = {6, 16, 31, 61, 91, 121, 151, 181, 366, 721}

    padroes = [
        (r"^1-(\d+)", lambda m: f"0-{m.group(1)} dias"),

        (r"^de\s*(\d+)\s*a\s*(\d+)\s*dias?$", lambda m:
         f"{int(m.group(1)) - 1 if int(m.group(1)) in subtrair_se_comecar_em else m.group(1)}-{m.group(2)} dias"),

        (r"^(\d+)\s*(?:a|e|-)\s*(\d+)\s*dias?$", lambda m:
         f"{int(m.group(1)) - 1 if int(m.group(1)) in subtrair_se_comecar_em else m.group(1)}-{m.group(2)} dias"),

        (r"^(\d+)\s*(?:a|e|-)\s*(\d+)$", lambda m:
         f"{int(m.group(1)) - 1 if int(m.group(1)) in subtrair_se_comecar_em else m.group(1)}-{m.group(2)} dias"),

        (r"^até\s*(\d+)(?:\s*dias?)?$", lambda m: f"<= {m.group(1)} dias"),

        (r"^>\s*(\d+)$", lambda m: f"> {int(m.group(1))-1 if int(m.group(1)) == 121 else m.group(1)} dias"),

        (r"^(?:acima\s*de|superior(?:\s*a)?)\s*(\d+)(?:\s*dias?)?$", lambda m: f"> {int(m.group(1))-1 if int(m.group(1)) == 121 else m.group(1)} dias"),
    ]

    for padrao, formato in padroes:
        match = re.fullmatch(padrao, texto)
        if match:
            return formato(match)
    return None


In [200]:
def padronizar_colunas_dias(df):
    columns = pd.Series(df.columns)

    result = columns.apply(verificar_padrao)
    print(result)
    days_columns = columns[result]

    final = days_columns.apply(padronizar_string)

    columns[result] = final

    df.columns = columns

    return df

In [201]:
import yaml
def carregar_yaml(caminho_yaml):
    with open(caminho_yaml, 'r', encoding='utf-8') as f:
        dados = yaml.safe_load(f)

    return dados  # Retorna como dicionário: {coluna_final: [possibilidades]}

colunas_equivalentes = carregar_yaml("../YAMLs/colunas.YAML")
padroes_regex_col = carregar_yaml("../YAMLs/regex.YAML")

colunas_equivalentes

{'PL Total': ['Patrimônio Líquido Total', 'PL Total Classe (R$ mil)'],
 'PL Sênior': ['PL Senior', 'PL Senior ', 'PL Sênior '],
 'PL Mezanino': ['PL Subordinada Mezanino'],
 'PL Subordinada Jr': ['PL Subordinada Jr.',
  'PL Subordinada Jr:',
  'PL Subordinada Junior',
  'PL Subordinada Júnior'],
 'Carteira (Direitos Creditórios)': ['Carteira (Direiros Creditórios)',
  'Carteira Direitos Creditórios (Vlr Presente)',
  'Carteira Total (Direitos Creditórios)',
  'Direitos Creditórios',
  'Total Direitos Creditórios (R$ mil)'],
 'Quantidade de Cedentes': ['Qtidade Cedentes',
  'Quantidade Cedentes',
  '# cedentes'],
 'Quantidade de Sacados': ['Qtidade Sacados',
  'Quantidade Sacados',
  '# sacados'],
 'Caixa/Disponibilidades': ['Caixa+Disponibilidades'],
 'Taxa Média': ['TIR - Mês -  Carteira (Dir.Creditórios)',
  'Taxa Média Recebíveis Média ponderada',
  'Taxa Média a.m',
  'Taxa Média a.m (*)',
  'Taxa Média (a.m.)',
  'Taxa Ponderada de Cessão'],
 'Prazo Médio': ['Prazo Médio Carteira 

In [202]:
def renomear_colunas_equivalentes(df, colunas_equivalentes_dict):
    renomear_dict = {}
    for nome_padrao, alternativas in colunas_equivalentes_dict.items():
        if alternativas is None: continue
        for nome in alternativas:
            if nome in df.columns:
                renomear_dict[nome] = nome_padrao
    return df.rename(columns=renomear_dict)

In [203]:
def verificar_padrao2(entry):
    text = str(entry).lower().strip()
    padroes = [
        r'(\d+)\s*-\s*(\d+)\s*dias?',                   # intervalo: 10 - 20 dias
        r'>\s*(\d+)\s*dias?',                           # > 120 dias
        r'<=\s*(\d+)\s*dias?'                           # <= 10 dias
    ]
    return any(re.search(p, text) for p in padroes)

def padronizar_colunas_dias2(df):
    columns = pd.Series(df.columns)
    result = columns.apply(verificar_padrao2)  # vetor booleano, ex: [True, False, False, ...]
    last_true_idx = None
    new_columns = []

    for idx, col_name in enumerate(columns):
        if not result.iloc[idx]:
            new_columns.append(col_name)
            last_true_idx = idx
        else:
            if last_true_idx is not None:
                new_name = f"({columns.iloc[last_true_idx]}){col_name}"
                new_columns.append(new_name)
            else:
                new_columns.append(col_name)
    df.columns = new_columns
    return df

In [204]:
import os

# Caminho da pasta com os arquivos CSV
caminho_pasta = '../out'

# Dicionário para armazenar os DataFrames
csv_dict = {}

def remover_sufixos(colunas):
    colunas_novas = []
    for col in colunas:
        # Remove sufixo do tipo .1, .2, .3 no final da string
        col_nova = re.sub(r'\.\d+$', '', col)
        colunas_novas.append(col_nova)
    return colunas_novas

# Percorre todos os arquivos na pasta
for arquivo in os.listdir(caminho_pasta):
    if arquivo.endswith('.csv'):
        caminho_arquivo = os.path.join(caminho_pasta, arquivo)
        nome_arquivo = os.path.splitext(arquivo)[0]  # Remove a extensão
        df = pd.read_csv(caminho_arquivo, sep=';', encoding='utf-8-sig', index_col = "Data", ).astype("float64")
        # ajuste encoding se necessário
        print(nome_arquivo)
        df.columns = remover_sufixos(df.columns)
        df = padronizar_colunas_dias(df)
        df = renomear_colunas_equivalentes(df, colunas_equivalentes)
        df = padronizar_colunas_dias2(df)
        csv_dict[nome_arquivo] = df

ALFA
0     False
1     False
2     False
3     False
4     False
      ...  
67    False
68    False
69    False
70    False
71    False
Length: 72, dtype: bool
APPALOOSA
0     False
1     False
2     False
3     False
4     False
      ...  
65    False
66    False
67    False
68    False
69    False
Length: 70, dtype: bool
ARTICO
0     False
1     False
2     False
3     False
4     False
      ...  
73    False
74    False
75    False
76    False
77    False
Length: 78, dtype: bool
BARCELONA
0     False
1     False
2     False
3     False
4     False
      ...  
61    False
62    False
63    False
64    False
65    False
Length: 66, dtype: bool
MULTIASSET
0     False
1     False
2     False
3     False
4     False
      ...  
88    False
89    False
90    False
91    False
92    False
Length: 93, dtype: bool
MULTIPLIKE
0     False
1     False
2     False
3     False
4     False
      ...  
65    False
66    False
67    False
68    False
69    False
Length: 70, dtype: bool
ONE7
0    

In [205]:
from collections import Counter

contador_colunas = Counter()

for df in csv_dict.values():
    contador_colunas.update(df.columns.tolist())

colunas_ordenadas = dict(
    sorted(
        contador_colunas.items(),
        key=lambda x: (x[0] is None, str(x[0]))
    )
)

print(len(colunas_ordenadas))
for coluna, qtd in colunas_ordenadas.items():
    print(f"{coluna}: {qtd}")

237
$ Médio por cedente (*): 1
$ Médio por operação : 1
% Recompra: 1
% Subordinação: 1
% Total Direitos Creditórios sobre PL: 1
% do PL A Vencer: 1
% do PL FIDC Investido: 1
% do PL PDD: 1
% do PL Subordinada Júnior: 1
% do PL Subordinada Mezanino: 1
% do PL Subordinada Mezanino + Júnior: 1
% do PL Sênior: 1
% do PL Vencidos: 1
% do PL da Subordinada Júnior / PL Subordinada Meza + Jr.: 1
% do Volume liquidado antecipadamente: 1
% do Volume liquidado no prazo: 1
% do Volume liquidado recompra: 1
% do Volume liquidado vencido: 1
(% Total Direitos Creditórios sobre PL)0-5 dias: 2
(% Total Direitos Creditórios sobre PL)30-60 dias: 2
(% Total Direitos Creditórios sobre PL)5-30 dias: 2
(% Total Direitos Creditórios sobre PL)60-90 dias: 2
(% Total Direitos Creditórios sobre PL)90-120 dias: 2
(% Total Direitos Creditórios sobre PL)> 120 dias: 2
(Efeito Vagão)0-5 dias: 1
(Efeito Vagão)30-60 dias: 1
(Efeito Vagão)5-30 dias: 1
(Efeito Vagão)60-90 dias: 1
(Efeito Vagão)90-120 dias: 1
(Efeito Vagã

In [206]:
colunas_ordenadas = dict(
    sorted(
        contador_colunas.items(),
        key=lambda x: x[1],  # ordenar pelo valor (quantidade)
        reverse=True         # maior para menor
    )
)

colunas_ordenadas

{'PL Total': 10,
 'PL Sênior': 10,
 'PL Mezanino': 10,
 'PL Subordinada Jr': 10,
 'Carteira (Direitos Creditórios)': 10,
 'Taxa Média': 10,
 'Prazo Médio': 10,
 'Vencidos Total': 10,
 'PDD Total': 10,
 'Recompra (R$)': 10,
 'Cedente 1': 10,
 'Cedente 2': 10,
 'Cedente 3': 10,
 'Cedente 4': 10,
 'Cedente 5': 10,
 'Cedente 6': 10,
 'Cedente 7': 10,
 'Cedente 8': 10,
 'Cedente 9': 10,
 'Cedente 10': 10,
 'Sacado 1': 10,
 'Sacado 2': 10,
 'Sacado 3': 10,
 'Sacado 4': 10,
 'Sacado 5': 10,
 'Sacado 6': 10,
 'Sacado 7': 10,
 'Sacado 8': 10,
 'Sacado 9': 10,
 'Sacado 10': 10,
 'Concentração Top 10 Sacados (R$)': 10,
 'Concentração Top 10 Cedentes (R$)': 10,
 'Duplicata': 9,
 'Cheques': 9,
 'Caixa/Disponibilidades': 9,
 'Quantidade de Cedentes': 9,
 'Quantidade de Sacados': 9,
 'Volume Operado': 8,
 'Liquidado Total(R$)': 8,
 'Retorno Cota Subordinada': 8,
 'Contrato': 8,
 '(Vencidos Total)30-60 dias': 7,
 '(Vencidos Total)60-90 dias': 7,
 '(Vencidos Total)90-120 dias': 7,
 '(PDD Total)30-60 di

In [207]:
for key in csv_dict.keys():
    print(key)
    print(csv_dict[key].columns)

ALFA
Index(['PL Total', 'PL Sênior', 'PL Mezanino', 'PL Subordinada Jr',
       'Carteira Total (Recebíveis)', 'Carteira (Direitos Creditórios)',
       'Duplicata', 'Cheques', 'Carteira (Outros - Anima)',
       'Percentual Direitos Creditórios/PL', 'Caixa/Disponibilidades',
       'Fundo Soberano', 'Quantidade de Cedentes', 'Valor médio por cedente',
       'Quantidade de Sacados', 'Quantidade de títulos em aberto (Anima)',
       '$ Médio por operação ', 'Taxa Média', 'Nivel de confirmação media',
       'Prazo Médio', 'Volume Operado', 'Liquidado Total(R$)',
       'Vencidos Total', '(Vencidos Total)0-15 dias',
       '(Vencidos Total)15-30 dias', '(Vencidos Total)30-60 dias',
       '(Vencidos Total)60-90 dias', '(Vencidos Total)90-120 dias',
       '(Vencidos Total)120-150 dias', '(Vencidos Total)150-180 dias',
       '(Vencidos Total)180-999 dias', 'Percentual de vencidos total (PL)',
       'PDD Total', '(PDD Total)0-15 dias', '(PDD Total)15-30 dias',
       '(PDD Total)30-60 d

In [208]:
colunas_equivalentes.keys()

dict_keys(['PL Total', 'PL Sênior', 'PL Mezanino', 'PL Subordinada Jr', 'Carteira (Direitos Creditórios)', 'Quantidade de Cedentes', 'Quantidade de Sacados', 'Caixa/Disponibilidades', 'Taxa Média', 'Prazo Médio', 'Ticket Médio', 'Retorno Cota Subordinada', 'Cheques', 'Duplicata', 'Duplicata de Serviço Físico', 'Contrato', 'Nota Comercial', 'Confissão de Dívida', 'Nota Promissória', 'CCB', 'Resgates Subordinada Jr.', 'Resgates Subordinada Mezanino', 'Resgates Sênior', 'Vencidos Total', 'PDD Total', 'PDD À Vencer', 'Liquidado Total(R$)', 'Liquidado Antecipadamente', 'Liquidado no Prazo', 'Liquidado Vencido', 'Entre D1-D5', 'Entre D1-D15', 'Entre D6-D15', 'Entre D16-D30', 'Acima de D30', 'Entre D31-D90', 'Entre D91-D120', 'Acima de D120', 'Recompra (R$)', 'Concentração Top 10 Cedentes (R$)', 'Concentração Top 10 Sacados (R$)', 'Número de Funcionários', 'Volume Operado'])

In [209]:
def selecionar_colunas_por_nome(dataframes, padroes_colunas):
    # Usa apenas as chaves como nomes exatos de colunas a buscar
    nomes_colunas_procuradas = list(padroes_colunas.keys())
    print("Liquidado Total(R$)" in padroes_colunas)
    print(nomes_colunas_procuradas)
    resultado = {}

    for nome_df, df in dataframes.items():
        colunas_encontradas = []
        for col in df.columns:
            if col in nomes_colunas_procuradas:
                colunas_encontradas.append(col)
        resultado[nome_df] = colunas_encontradas

    return resultado

def selecionar_colunas_por_regex(dataframes, padroes_colunas):
    teste = list(padroes_colunas.values())[0]

    padroes_compilados = [re.compile(p) for p in teste]

    resultado = {}

    for nome_df, df in dataframes.items():
        colunas_capturadas = []
        for col in df.columns:
            for padrao in padroes_compilados:
                if padrao.fullmatch(str(col)):
                    colunas_capturadas.append(col)
                    break
        resultado[nome_df] = colunas_capturadas

    return resultado

list1 = selecionar_colunas_por_nome(csv_dict, colunas_equivalentes)
list2 = selecionar_colunas_por_regex(csv_dict, padroes_regex_col)

True
['PL Total', 'PL Sênior', 'PL Mezanino', 'PL Subordinada Jr', 'Carteira (Direitos Creditórios)', 'Quantidade de Cedentes', 'Quantidade de Sacados', 'Caixa/Disponibilidades', 'Taxa Média', 'Prazo Médio', 'Ticket Médio', 'Retorno Cota Subordinada', 'Cheques', 'Duplicata', 'Duplicata de Serviço Físico', 'Contrato', 'Nota Comercial', 'Confissão de Dívida', 'Nota Promissória', 'CCB', 'Resgates Subordinada Jr.', 'Resgates Subordinada Mezanino', 'Resgates Sênior', 'Vencidos Total', 'PDD Total', 'PDD À Vencer', 'Liquidado Total(R$)', 'Liquidado Antecipadamente', 'Liquidado no Prazo', 'Liquidado Vencido', 'Entre D1-D5', 'Entre D1-D15', 'Entre D6-D15', 'Entre D16-D30', 'Acima de D30', 'Entre D31-D90', 'Entre D91-D120', 'Acima de D120', 'Recompra (R$)', 'Concentração Top 10 Cedentes (R$)', 'Concentração Top 10 Sacados (R$)', 'Número de Funcionários', 'Volume Operado']


In [210]:
#resultado = list1
resultado = {k: list1.get(k, []) + list2.get(k, []) for k in list1.keys()}

resultado

{'ALFA': ['PL Total',
  'PL Sênior',
  'PL Mezanino',
  'PL Subordinada Jr',
  'Carteira (Direitos Creditórios)',
  'Duplicata',
  'Cheques',
  'Caixa/Disponibilidades',
  'Quantidade de Cedentes',
  'Quantidade de Sacados',
  'Taxa Média',
  'Prazo Médio',
  'Volume Operado',
  'Liquidado Total(R$)',
  'Vencidos Total',
  'PDD Total',
  'PDD À Vencer',
  'Recompra (R$)',
  'Retorno Cota Subordinada',
  'Número de Funcionários',
  'Concentração Top 10 Sacados (R$)',
  'Concentração Top 10 Cedentes (R$)',
  '(Vencidos Total)0-15 dias',
  '(Vencidos Total)15-30 dias',
  '(Vencidos Total)30-60 dias',
  '(Vencidos Total)60-90 dias',
  '(Vencidos Total)90-120 dias',
  '(Vencidos Total)120-150 dias',
  '(Vencidos Total)150-180 dias',
  '(Vencidos Total)180-999 dias',
  '(PDD Total)0-15 dias',
  '(PDD Total)15-30 dias',
  '(PDD Total)30-60 dias',
  '(PDD Total)60-90 dias',
  '(PDD Total)90-120 dias',
  '(PDD Total)120-150 dias',
  '(PDD Total)150-180 dias',
  '(PDD Total)180-999 dias',
  'Ced

In [211]:
colunas_equivalentes.keys()

dict_keys(['PL Total', 'PL Sênior', 'PL Mezanino', 'PL Subordinada Jr', 'Carteira (Direitos Creditórios)', 'Quantidade de Cedentes', 'Quantidade de Sacados', 'Caixa/Disponibilidades', 'Taxa Média', 'Prazo Médio', 'Ticket Médio', 'Retorno Cota Subordinada', 'Cheques', 'Duplicata', 'Duplicata de Serviço Físico', 'Contrato', 'Nota Comercial', 'Confissão de Dívida', 'Nota Promissória', 'CCB', 'Resgates Subordinada Jr.', 'Resgates Subordinada Mezanino', 'Resgates Sênior', 'Vencidos Total', 'PDD Total', 'PDD À Vencer', 'Liquidado Total(R$)', 'Liquidado Antecipadamente', 'Liquidado no Prazo', 'Liquidado Vencido', 'Entre D1-D5', 'Entre D1-D15', 'Entre D6-D15', 'Entre D16-D30', 'Acima de D30', 'Entre D31-D90', 'Entre D91-D120', 'Acima de D120', 'Recompra (R$)', 'Concentração Top 10 Cedentes (R$)', 'Concentração Top 10 Sacados (R$)', 'Número de Funcionários', 'Volume Operado'])

In [234]:
def selecionando_colunas_final(csv_dict, colunas):
    for key in csv_dict.keys():
        csv_dict[key] = csv_dict[key][colunas[key]]
    return csv_dict

print(resultado)
teste_final = selecionando_colunas_final(csv_dict, resultado)
print(teste_final)

{'ALFA': ['PL Total', 'PL Sênior', 'PL Mezanino', 'PL Subordinada Jr', 'Carteira (Direitos Creditórios)', 'Duplicata', 'Cheques', 'Caixa/Disponibilidades', 'Quantidade de Cedentes', 'Quantidade de Sacados', 'Taxa Média', 'Prazo Médio', 'Volume Operado', 'Liquidado Total(R$)', 'Vencidos Total', 'PDD Total', 'PDD À Vencer', 'Recompra (R$)', 'Retorno Cota Subordinada', 'Número de Funcionários', 'Concentração Top 10 Sacados (R$)', 'Concentração Top 10 Cedentes (R$)', '(Vencidos Total)0-15 dias', '(Vencidos Total)15-30 dias', '(Vencidos Total)30-60 dias', '(Vencidos Total)60-90 dias', '(Vencidos Total)90-120 dias', '(Vencidos Total)120-150 dias', '(Vencidos Total)150-180 dias', '(Vencidos Total)180-999 dias', '(PDD Total)0-15 dias', '(PDD Total)15-30 dias', '(PDD Total)30-60 dias', '(PDD Total)60-90 dias', '(PDD Total)90-120 dias', '(PDD Total)120-150 dias', '(PDD Total)150-180 dias', '(PDD Total)180-999 dias', 'Cedente 1', 'Cedente 2', 'Cedente 3', 'Cedente 4', 'Cedente 5', 'Cedente 6', 'C

In [213]:
def juntar_por_data(csv_dict, data):
    lista_dados = []

    for key, df in csv_dict.items():
        if data in df.index:
            linha = df.loc[data]
            linha_df = pd.DataFrame([linha], index=[key])
        else:
            linha_df = pd.DataFrame([{col: pd.NA for col in df.columns}], index=[key])

        lista_dados.append(linha_df)

    resultado = pd.concat(lista_dados, axis=0, sort=True)
    return resultado

In [214]:
# checar dps esse Taxa Média Recebíveis Média ponderada

#o padrão vai ser "Prazo Médio"
# checar dps esse Prazo Médio Recebíveis Estoque (dias úteis) e Prazo médio aquisição no mês
# formatar todas de dias corridos para úteis, se for o caso
# oq não tem especificando é corrido

#remover recebíveis

# hoje a tarde fazer:
    # ajeitar TUDO do SOLAR
    # ajeitar todas as %
    # adicionar últimos dados gerais
    # ai dps disso eu vejo mais oq adicionar e também os q ficarão no fim do DF final
    # também tenho que lembrar que não dá pra unir tudo de uma vez então pensar em como fica a estrutura final
    #    - vou manter os dataframes PARSEADOS em CSV padronizados salvos separados
    #    - manter o planilhão por mês
    # ajeitar os 0-15 que importam, colocando os padrãozinho lá

In [215]:
def padronizar_indices_para_fim_do_mes(csv_dict):
    csv_dict_padronizado = {}

    for key, df in csv_dict.items():
        df = df.copy()

        # Garante que o índice está no formato datetime
        df.index = pd.to_datetime(df.index, errors='coerce')

        # Remove linhas com índice inválido (caso existam)
        df = df[~df.index.isna()]

        # Ajusta cada data para o último dia do mês
        df.index = df.index.to_period('M').to_timestamp('M')

        # Se houver duplicação após essa conversão, agrupar (aqui usamos média, mas pode ser alterado)
        df = df.groupby(df.index).first()

        csv_dict_padronizado[key] = df

    return csv_dict_padronizado

teste_final = padronizar_indices_para_fim_do_mes(teste_final)

In [216]:
resultado_quasefinal = juntar_por_data(teste_final, "2025-04-30")

In [217]:
def mostrar_colunas_repetidas(csv_dict):
    for key, df in csv_dict.items():
        colunas_repetidas = df.columns[df.columns.duplicated()]
        if len(colunas_repetidas) > 0:
            print(f'DataFrame "{key}" tem colunas repetidas:')
            print(colunas_repetidas.tolist())
        else:
            print(f'DataFrame "{key}" não tem colunas repetidas.')

# Exemplo de uso:
mostrar_colunas_repetidas(teste_final)

DataFrame "ALFA" não tem colunas repetidas.
DataFrame "APPALOOSA" não tem colunas repetidas.
DataFrame "ARTICO" não tem colunas repetidas.
DataFrame "BARCELONA" não tem colunas repetidas.
DataFrame "MULTIASSET" não tem colunas repetidas.
DataFrame "MULTIPLIKE" não tem colunas repetidas.
DataFrame "ONE7" não tem colunas repetidas.
DataFrame "PAGOL" não tem colunas repetidas.
DataFrame "PHD" não tem colunas repetidas.
DataFrame "SOLAR" não tem colunas repetidas.


In [218]:
teste_final.keys()

dict_keys(['ALFA', 'APPALOOSA', 'ARTICO', 'BARCELONA', 'MULTIASSET', 'MULTIPLIKE', 'ONE7', 'PAGOL', 'PHD', 'SOLAR'])

In [219]:
colunas_equivalentes.keys()

dict_keys(['PL Total', 'PL Sênior', 'PL Mezanino', 'PL Subordinada Jr', 'Carteira (Direitos Creditórios)', 'Quantidade de Cedentes', 'Quantidade de Sacados', 'Caixa/Disponibilidades', 'Taxa Média', 'Prazo Médio', 'Ticket Médio', 'Retorno Cota Subordinada', 'Cheques', 'Duplicata', 'Duplicata de Serviço Físico', 'Contrato', 'Nota Comercial', 'Confissão de Dívida', 'Nota Promissória', 'CCB', 'Resgates Subordinada Jr.', 'Resgates Subordinada Mezanino', 'Resgates Sênior', 'Vencidos Total', 'PDD Total', 'PDD À Vencer', 'Liquidado Total(R$)', 'Liquidado Antecipadamente', 'Liquidado no Prazo', 'Liquidado Vencido', 'Entre D1-D5', 'Entre D1-D15', 'Entre D6-D15', 'Entre D16-D30', 'Acima de D30', 'Entre D31-D90', 'Entre D91-D120', 'Acima de D120', 'Recompra (R$)', 'Concentração Top 10 Cedentes (R$)', 'Concentração Top 10 Sacados (R$)', 'Número de Funcionários', 'Volume Operado'])

In [220]:
from itertools import chain

lista_unica = pd.Series(chain.from_iterable(list2.values()))

testeee = list(lista_unica.drop_duplicates())

In [221]:
def extrair_ordem(item):
    match = re.match(r'^\((.*?)\)\s*(>?=?\s*\d+)(?:\s*-\s*(\d+))?', item)
    if match:
        grupo = match.group(1)
        ini_raw = match.group(2)
        fim_raw = match.group(3)

        ini = int(re.sub(r'\D', '', ini_raw)) if ini_raw else 0
        fim = int(fim_raw) if fim_raw else (ini if '>' not in ini_raw else 9999)

        return (grupo, ini, fim)
    else:
        return ('ZZZ', 99999, 99999)  # Ficam no fim: 'Cedente', 'Sacado' etc.

In [222]:
# Ordenar
#lista_ordenada = sorted(testeee, key=extrair_ordem)

#lista_ordenada

In [223]:
lista_unica2 = list(colunas_equivalentes.keys())

#ordem_final = lista_unica2 +lista_ordenada

In [224]:
#ordem_final
# resolver problemas das colunas repetidas

In [225]:
import re

def organizar_colunas_df(lista_unica2, testeee):
    def extrair_ordem(item):
        match = re.match(r'^\((.*?)\)\s*(>?=?\s*\d+)(?:\s*-\s*(\d+))?', item)
        if match:
            grupo = match.group(1)
            ini_raw = match.group(2)
            fim_raw = match.group(3)

            ini = int(re.sub(r'\D', '', ini_raw)) if ini_raw else 0
            fim = int(fim_raw) if fim_raw else (ini if '>' not in ini_raw else 9999)

            return (grupo, ini, fim)
        else:
            return ('ZZZ', 99999, 99999)  # Para colunas fora do padrão

    # Ordena colunas do tipo (Grupo)X-Y dias
    lista_ordenada = sorted(testeee, key=extrair_ordem)
    resultado = lista_unica2 + lista_ordenada

    # Regras de reorganização
    if 'Concentração Top 10 Cedentes (R$)' in resultado and 'Cedente 1' in resultado:
        resultado.remove('Concentração Top 10 Cedentes (R$)')
        idx = resultado.index('Cedente 1')
        resultado.insert(idx, 'Concentração Top 10 Cedentes (R$)')

    if 'Concentração Top 10 Sacados (R$)' in resultado and 'Sacado 1' in resultado:
        resultado.remove('Concentração Top 10 Sacados (R$)')
        idx = resultado.index('Sacado 1')
        resultado.insert(idx, 'Concentração Top 10 Sacados (R$)')

    padrao_grupo = re.compile(r'^\(([^)]+)\)\s*[\d>]')
    grupos = {}
    for i, col in enumerate(resultado):
        match = padrao_grupo.match(col)
        if match:
            grupo = match.group(1)
            if grupo not in grupos and grupo in resultado:
                grupos[grupo] = i

    for grupo, idx_ref in grupos.items():
        if grupo in resultado:
            resultado.remove(grupo)
            resultado.insert(idx_ref - 1, grupo)

    # Mover 'PDD À Vencer' após último '(PDD Total)X-Y dias'
    if 'PDD À Vencer' in resultado:
        padrao_pdd = re.compile(r'^\(PDD Total\)\s*\d+\s*-\s*\d+\s*dias')
        indices_pdd = [i for i, col in enumerate(resultado) if padrao_pdd.match(col)]
        if indices_pdd:
            idx_destino = max(indices_pdd)
            resultado.remove('PDD À Vencer')
            resultado.insert(idx_destino, 'PDD À Vencer')

    return resultado

In [226]:
ordem_final = organizar_colunas_df(lista_unica2, testeee)

In [227]:
# valores dos tipos de ativos financeiros

resultado_quasefinal[ordem_final]

,PL Total,PL Sênior,PL Mezanino,PL Subordinada Jr,Carteira (Direitos Creditórios),Quantidade de Cedentes,Quantidade de Sacados,Caixa/Disponibilidades,Taxa Média,Prazo Médio,...,Sacado 1,Sacado 2,Sacado 3,Sacado 4,Sacado 5,Sacado 6,Sacado 7,Sacado 8,Sacado 9,Sacado 10
ALFA,5.598506e+07,2.199191e+07,0.000000e+00,3.399315e+07,5.307486e+07,82.0,3431.0,48438.44,0.022500,55.498530,...,1133381.51,904865.38,899158.06,876366.42,864658.07,838767.35,776919.78,755725.97,753557.34,749626.89
APPALOOSA,1.001210e+08,5.976211e+07,2.015185e+07,2.020706e+07,9.085485e+07,159.0,1795.0,12919151.00,0.024300,38.340000,...,2984199.11,2929089.24,2559215.76,2088988.89,2026239.89,1954596.66,1825106.54,1751017.27,1619262.22,1598968.32
ARTICO,2.698650e+08,1.576472e+08,4.759298e+07,6.462484e+07,2.362079e+08,84.0,1058.0,32158205.46,0.017219,87.000000,...,10961976.21,8207358.80,6790398.53,4117255.10,3477036.45,3452719.16,3252754.03,2898932.10,2450243.81,2146715.45
BARCELONA,9.827086e+07,4.823815e+07,1.717214e+07,3.286056e+07,9.147797e+07,175.0,4039.0,7428303.03,0.028800,45.600000,...,4322497.28,2801970.76,1911557.19,1859666.77,1824915.62,1290029.83,1254892.33,1218931.53,1212395.09,1014758.43
MULTIASSET,3.129984e+08,7.358007e+07,7.298921e+07,1.664291e+08,2.767261e+08,99.0,9649.0,1098.16,0.020900,151.050010,...,16506920.85,10563001.52,10299250.06,10233916.98,9079514.82,8296792.37,6801516.90,6718595.39,6627403.16,6442768.72
MULTIPLIKE,1.167739e+09,6.996966e+08,2.333193e+08,2.347235e+08,1.081891e+09,NaN,NaN,3578768.69,0.020100,92.510000,...,29060526.51,20876433.33,20050187.23,19536280.14,19260990.78,19140068.29,19050280.11,16759595.02,16332189.32,15430254.00
ONE7,2.795621e+08,1.576532e+08,2.217392e+07,9.973501e+07,2.701015e+08,763.0,6636.0,NaN,0.026800,23.780000,...,6276039.99,4413772.00,4197021.13,4089931.49,3450855.46,3339737.08,3192687.54,2993015.78,2758067.43,2683623.49
PAGOL,6.166202e+07,1.197339e+07,0.000000e+00,4.968863e+07,6.882882e+07,2.0,6610.0,5501335.77,0.016552,282.858882,...,86808.59,76511.01,75748.85,72991.80,67231.44,65691.88,65273.94,64372.18,63798.42,62905.78
PHD,1.829639e+08,1.092947e+08,3.633387e+07,3.733539e+07,1.693614e+08,154.0,6265.0,15861812.33,0.022974,129.136437,...,4106189.90,4082969.61,3915270.94,3269730.66,3216040.38,2919589.98,2871364.27,2632528.40,1834455.23,1737043.48
SOLAR,1.234262e+08,8.156198e+07,1.303532e+07,2.882895e+07,1.174088e+08,32.0,579.0,5637403.54,0.023876,28.902024,...,7076750.47,4992126.41,4050580.42,2958487.69,2807804.32,2672698.40,2535911.00,2401315.20,2174519.29,1808588.42


In [228]:
resultado_quasefinal[ordem_final].columns

Index(['PL Total', 'PL Sênior', 'PL Mezanino', 'PL Subordinada Jr',
       'Carteira (Direitos Creditórios)', 'Quantidade de Cedentes',
       'Quantidade de Sacados', 'Caixa/Disponibilidades', 'Taxa Média',
       'Prazo Médio', 'Ticket Médio', 'Retorno Cota Subordinada', 'Cheques',
       'Duplicata', 'Duplicata de Serviço Físico', 'Contrato',
       'Nota Comercial', 'Confissão de Dívida', 'Nota Promissória', 'CCB',
       'Resgates Subordinada Jr.', 'Resgates Subordinada Mezanino',
       'Resgates Sênior', 'Liquidado Total(R$)', 'Liquidado Antecipadamente',
       'Liquidado no Prazo', 'Liquidado Vencido', 'Entre D1-D5',
       'Entre D1-D15', 'Entre D6-D15', 'Entre D16-D30', 'Acima de D30',
       'Entre D31-D90', 'Entre D91-D120', 'Acima de D120', 'Recompra (R$)',
       'Número de Funcionários', 'Volume Operado', 'PDD Total',
       '(PDD Total)0-5 dias', '(PDD Total)0-15 dias', '(PDD Total)0-30 dias',
       '(PDD Total)5-30 dias', '(PDD Total)15-30 dias',
       '(PDD Total)3

In [229]:
#checar tipos e informar algumas coisas depois

testeee

['(Vencidos Total)0-15 dias',
 '(Vencidos Total)15-30 dias',
 '(Vencidos Total)30-60 dias',
 '(Vencidos Total)60-90 dias',
 '(Vencidos Total)90-120 dias',
 '(Vencidos Total)120-150 dias',
 '(Vencidos Total)150-180 dias',
 '(Vencidos Total)180-999 dias',
 '(PDD Total)0-15 dias',
 '(PDD Total)15-30 dias',
 '(PDD Total)30-60 dias',
 '(PDD Total)60-90 dias',
 '(PDD Total)90-120 dias',
 '(PDD Total)120-150 dias',
 '(PDD Total)150-180 dias',
 '(PDD Total)180-999 dias',
 'Cedente 1',
 'Cedente 2',
 'Cedente 3',
 'Cedente 4',
 'Cedente 5',
 'Cedente 6',
 'Cedente 7',
 'Cedente 8',
 'Cedente 9',
 'Cedente 10',
 'Sacado 1',
 'Sacado 2',
 'Sacado 3',
 'Sacado 4',
 'Sacado 5',
 'Sacado 6',
 'Sacado 7',
 'Sacado 8',
 'Sacado 9',
 'Sacado 10',
 '(Vencidos Total)0-30 dias',
 '(Vencidos Total)> 120 dias',
 '(PDD Total)0-30 dias',
 '(PDD Total)> 120 dias',
 '(Vencidos Total)0-5 dias',
 '(Vencidos Total)5-30 dias',
 '(PDD Total)0-5 dias',
 '(PDD Total)5-30 dias',
 '(PDD Total)180-360 dias',
 '(Vencidos 

In [230]:
# o que preciso calcular:
    # (mezanino + junior) / total * 100
    # junior / total * 100
    # PDD / total * 100
    # Vencidos / total * 100
    # Liquidação / total * 100
    # Recompra / total * 100
    # Duplicatas / total * 100
    # Valor Total Operado / total * 100

In [231]:
resultado_quasefinal[ordem_final] = resultado_quasefinal[ordem_final].astype('float64')

In [232]:
df = resultado_quasefinal
col_total = "PL Total"
col_mezanino = "PL Mezanino"
col_junior = "PL Subordinada Jr"
col_pdd = "PDD Total"
col_vencidos = "Vencidos Total"
col_liquidacao = "Liquidado Total(R$)"
col_recompra = "Recompra (R$)"
col_duplicatas = "Duplicata"
col_valor_total_operado = "Volume Operado"

df_result = pd.DataFrame(index=df.index)

df_result["(Mezanino + Junior) / PL Total (%)"] = (df[col_mezanino].fillna(0) + df[col_junior].fillna(0)) / df[col_total] * 100
df_result["PL Subordinada Jr / PL Total (%)"] = df[col_junior] / df[col_total] * 100
df_result["PDD Total / PL Total (%)"] = df[col_pdd] / df[col_total] * 100
df_result["Vencidos Total / PL Total (%)"] = df[col_vencidos] / df[col_total] * 100
df_result["Liquidado Total(R$) / PL Total (%)"] = df[col_liquidacao] / df[col_total] * 100
df_result["Recompra (R$) / PL Total (%)"] = df[col_recompra] / df[col_total] * 100
df_result["Duplicata / PL Total (%)"] = df[col_duplicatas] / df[col_total] * 100
df_result["Volume Operado / PL Total (%)"] = df[col_valor_total_operado] / df[col_total] * 100

df_result

,(Mezanino + Junior) / PL Total (%),PL Subordinada Jr / PL Total (%),PDD Total / PL Total (%),Vencidos Total / PL Total (%),Liquidado Total(R$) / PL Total (%),Recompra (R$) / PL Total (%),Duplicata / PL Total (%),Volume Operado / PL Total (%)
ALFA,60.718258,60.718258,2.743012,7.894622,34.571699,1.599662,88.566323,49.612488
APPALOOSA,40.310130,20.182638,3.793427,3.717571,45.813403,2.326105,54.434369,39.058734
ARTICO,41.582944,23.947098,0.025490,1.613715,43.678885,1.162079,38.886182,50.490095
BARCELONA,50.913067,33.438769,7.537722,11.548592,55.195336,1.672575,63.191766,NaN
MULTIASSET,76.491867,53.172508,2.785524,6.403716,23.330749,3.448975,87.874480,24.656593
MULTIPLIKE,40.081104,20.100675,0.018631,3.505963,41.621353,0.613988,38.214505,41.316179
ONE7,43.607096,35.675435,5.077351,6.231930,NaN,8.370737,67.801518,NaN
PAGOL,80.582232,80.582232,20.482322,8.526370,NaN,0.000000,0.000000,0.233805
PHD,40.264357,20.405870,1.885314,9.014293,50.207870,2.062356,56.882198,51.738233
SOLAR,33.918450,23.357229,-0.323269,3.637098,22.717236,0.901507,NaN,25.306889
